In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pickle
import os

from pandas_helpers import *
from branches import *
from tqdm import tqdm

import sys
sys.path.append('/exp/sbnd/app/users/munjung/pyana_utils')
from particle_energies import *
from constants import *
from sbnd_configs import *
from flatcaf_utils import *

sys.path.append('/exp/sbnd/app/users/munjung/xsec/systs/cafpyana/makedf')
from chi2pid import *

plt.style.use('./presentation.mplstyle')

# ignore FutureWarning from uproot
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning) # df2 = df.sort_index()

# load hits from file

In [ ]:
testfile = "/exp/sbnd/data/users/munjung/xsec/NuINT2025/test_caf.root"
testfile = "/pnfs/sbn/data_add/sbn_nd/poms_production/mc/MCP2025B_1e20_05/v10_06_00_05/prodgenie_corsika_proton_rockbox_sbnd/CV/caf/7d/b0/caf.flat.caf-a742052d-531b-4d81-884f-1c82858c7a06.root"
f = uproot.open(testfile+":recTree")

In [ ]:
plane = 2
trkhitbranches = trkhitbranches_perplane(plane)
trkhitbranches += [
    trkbranch + "calo.%i.points.phi"% plane,
    trkbranch + "calo.%i.points.efield"% plane,
]

hitdf = loadbranches(f, trkhitbranches)
hitdf = hitdf.rec.slc.reco.pfp.trk.calo.I2.points


# load hits from hitfile

In [ ]:
df_list = []
output = "outputs/proton_hits_data.h5"
with pd.HDFStore(output) as hdf_pd:
    this_key = f"hits"
    df = hdf_pd.get(this_key)
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)
df.drop(columns=["index"], inplace=True)
df

In [ ]:
dqdx_corr = dqdx(df, gain="SBND", calibrate="SBND", isMC=False)
df["dqdx_uncorr"] = df.dqdx.copy()
df["dqdx"] = dqdx_corr

In [ ]:
df[["dqdx_uncorr", "dqdx"]]

In [ ]:
# save new DF

with pd.HDFStore("/exp/sbnd/data/users/munjung/calibration/proton_hits/2025B/proton_hits-Data-1e20-plane2-nogaincorr.h5") as hdf_pd:
    this_key = f"hits"
    hdf_pd.put(key=this_key, value=df, format="fixed")
    print(f"Saved {this_key}: {df.memory_usage(deep=True).sum() / (1024**3):.4f} GB")

with pd.HDFStore("/exp/sbnd/data/users/munjung/calibration/proton_hits/2025B/proton_hits-Data-1e20-plane2-nogaincorr.h5") as hdf_pd:
    this_key = f"hits"
    check_df = hdf_pd.get(this_key)

In [ ]:
rr_list = list(np.arange(1.2, 1.5, 0.3))
bins = np.linspace(0.25e4, 0.75e4, 41)
rr_list = list(np.arange(0.6, 1.5, 0.3))
bins = np.linspace(2e3, 0.9e4, 41)

bin_centers = (bins[:-1] + bins[1:]) / 2

# phi_list = [20,40,60,80,90]
phi_list = [0,90]
for ridx in range(len(rr_list)-1):
    for phi_idx in range(len(phi_list)-1):
        rr_lo = rr_list[ridx]
        rr_hi = rr_list[ridx+1]
        phi_lo = phi_list[phi_idx]
        phi_hi = phi_list[phi_idx+1]

        # data
        cut = (df.rr > rr_lo) & (df.rr < rr_hi) & (df.phi * 180/np.pi > phi_lo) & (df.phi * 180/np.pi < phi_hi)
        this_df = df[cut]
        n_data, bins = np.histogram(this_df.dqdx, bins=bins)
        color = plt.cm.viridis(ridx / len(rr_list))
        color = "black"

        # # MC
        # cut = (mc_df.rr > rr_lo) & (mc_df.rr < rr_hi) & (mc_df.phi * 180/np.pi > phi_lo) & (mc_df.phi * 180/np.pi < phi_hi)
        # this_mc_df = mc_df[cut]
        # # n_mc, bins = np.histogram(50 * this_mc_df.dqdx, bins=bins)
        # n_mc, bins = np.histogram(this_mc_df.dqdx, bins=bins)
        # # normalize to data
        # n_mc = n_mc * (n_data.sum() / n_mc.sum())

        # plot
        # plt.hist(bin_centers, bins, weights=n_mc, label="MC", alpha=0.5, color=color)
        plt.errorbar(bin_centers, n_data, yerr=np.sqrt(n_data), label="Data", fmt="o", capsize=3, color=color)

        plt.xlabel("dQ/dx")
        plt.ylabel("Hits (Normalized to Data)")
        plt.text(0.95, 0.7, "{:0.1f} cm < R.R. < {:0.1f} cm\n{:0.0f}$^\circ$ < $\phi$ < {:0.0f}$^\circ$".format(rr_lo, rr_hi, phi_lo, phi_hi),
                 fontsize=12, ha="right", va="top", transform=plt.gca().transAxes)
        plt.title("Proton Candidate")
        plt.legend()
        plt.show();